# 03 · Retrieval — Query 技巧（Rewrite / Multi-Query / HyDE）

目标：
- 复用课件里的“Query 改写”思路
- 在同一个 Chroma collection 上对比：
  - baseline（原始 query）
  - query rewrite（LLM 改写）
  - multi-query（生成多条查询，融合召回）
  - HyDE（先生成假设答案，再用它去检索）

> 依赖：上一节已写入 `data/chroma` + 设置 `OPENAI_API_KEY`。


In [4]:
pip install langchain_openai langchain_community

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp311-cp311-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.2-py3-none-any.whl (1.0 MB)
Using cached la

In [1]:
from __future__ import annotations

import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
embed_model =  os.getenv("EMBED_MODEL") # for Openrouter "qwen/qwen3-embedding-4b"
chat_model = os.getenv("CHAT_MODEL") 

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

SECTION_COLLECTION = "autel_annual_report_2024_sections"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}
print(embed_model)
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)

import chromadb
_chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

vs = Chroma(collection_name=COLLECTION, embedding_function=emb, client=_chroma_client)
section_vs = Chroma(collection_name=SECTION_COLLECTION, embedding_function=emb, client=_chroma_client)

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
print("ready — chunk collection:", COLLECTION, "| section collection:", SECTION_COLLECTION)
print("embed:", embed_model, "| chat:", chat_model, "| env:", ENV_FILE)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/opt/anaconda3/envs/voc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen/Qwen3-Embedding-8B
ready — chunk collection: autel_annual_report_2024 | section collection: autel_annual_report_2024_sections
embed: Qwen/Qwen3-Embedding-8B | chat: deepseek-ai/DeepSeek-V3.2 | env: /Users/mengbai/Documents/AI-training/.env


/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_57438/1367411917.py:55: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, client=_chroma_client)


### Baseline

In [27]:
# baseline 检索


QUESTION = "根据道通2024年年报来看，其核心竞争力在哪？"

hits = vs.similarity_search(QUESTION, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content.replace("\n", " "))
    print()


[1] {'doc_id': '道通24年年报__full', 'source': '道通24年年报', 'h1': '2024 年度报告', 'section_id': '道通24年年报__full::section_234', 'section_in_doc': 234, 'page_end': 306, 'doc_group': 'full_document', 'section_title': '(二) 担保情况', 'chunk_in_section': 12, 'parse_source': 'paddleocr_vl', 'page_start': 1, 'content_level': 'chunk', 'chunk_in_doc': 607, 'chunk_id': 607, 'source_doc_count': 306, 'h2': '(二) 担保情况', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md'}
。同时，由塞防科技及持有其股权的其他股东为公司提供反担保。2、2024年4月19日，公司2023年年度股东大会审议并通过了《关于2024年度对外担保额度预计的议案》，公司预计2024年度为控股子公司提供担保额度合计不超过人民币（或等值外币）4亿元，在上述预计的担保额度范围内，公司可根据实际情况对担保范围内的各子公司分配使用额度。</td></tr></table>

[2] {'chunk_in_doc': 0, 'doc_group': 'full_document', 'source_doc_count': 306, 'section_title': '2024 年度报告', 'doc_id': '道通24年年报__full', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'h1': '2024 年度报告', 'chunk_id': 0, 'parse_source': 'paddleocr_vl', 'section_in_doc': 0, 'page_start': 1, 'chunk_in_section': 0, 'page_end': 306, 'section_id': '道通24年年报__fu

In [20]:
### Query Rewrite

In [28]:
# 1) Query Rewrite：根据问题主题选择更贴近年报原文标题的检索锚点

from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Query Rewrite 模块。"
            "请先在心里识别用户问题主题，再输出 1 条更适合向量检索的中文 query。\n"
            "原则：\n"
            "1. 保留公司名、年份和核心问题，不要扩大或改变问题范围；\n"
            "2. 优先使用年报中可能真实出现的章节名、小节名、关键词锚点，而不是泛泛改写；\n"
            "3. 如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先考虑这类锚点：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势；\n"
            "4. 如果问题是其他主题，也按同样思路改写成更贴近年报标题的 query；\n"
            "5. 只输出 1 行 query，不要解释，不要回答问题。",
        ),
        ("human", "原始问题：{q}"),
    ]
)

rewritten = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
print("rewritten:\n", rewritten)

hits = vs.similarity_search(rewritten, k=10)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


rewritten:
 道通科技2024年主营业务分析及核心竞争力
[1] {'chunk_in_section': 12, 'chunk_in_doc': 272, 'doc_id': '道通24年年报__full', 'h1': '2024 年度报告', 'page_end': 306, 'section_in_doc': 87, 'chunk_id': 272, 'parse_source': 'paddleocr_vl', 'page_start': 1, 'content_level': 'chunk', 'section_id': '道通24年年报__full::section_87', 'section_title': '1、资产及负债状况', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'doc_group': 'full_document', 'source': '道通24年年报', 'source_doc_count': 306, 'h2': '1、资产及负债状况'}
style='text-align: center; word-wrap: break-word;'>-0.02</td><td style='text-align: center; word-wrap: break-word;'>2,868.01</td><td style='text-align: center; word-wrap: break-word;'>说明15</td></tr></table>

[2] {'content_level': 'chunk', 'section_in_doc': 448, 'doc_group': 'full_document', 'section_id': '道通24年年报__full::section_448', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'source': '道通24年年报', 'source_doc_count': 306, 'section_title': '(1). 无形资产情况', 'chunk_id': 1325, 'page_start': 1, 'chunk_i

### Multi-Query

In [29]:
# 2) Multi-Query：围绕同一问题生成“不同标题锚点”的多路检索 query

multi_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是面向中文年报检索的 Multi-Query 生成器。"
            "给定一个问题，请先识别问题主题，再输出 4 条中文 query，每条一行，不要编号，不要解释。\n"
            "这 4 条 query 必须分别覆盖以下 4 种角度：\n"
            "1. 原问题压缩版：保留公司、年份、核心主题；\n"
            "2. 章节标题版：优先使用年报里可能真实出现的章节/小节标题；\n"
            "3. 子主题展开版：把该主题拆成 2 到 4 个最可能回答问题的子点；\n"
            "4. 关键词聚合版：把公司、年份、主题词、近义词和标题锚点组合成一个更像检索式的 query；\n"
            "如果问题是‘核心竞争力/竞争优势/技术壁垒/护城河/领先性’，优先围绕这些词生成：报告期内核心竞争力分析、核心竞争力分析、技术创新优势、产品及解决方案优势、全球化本地化营销服务体系优势、全球化本地化产能及供应链优势、人才与团队优势。\n"
            "要求：\n"
            "- 保留公司名和年份；\n"
            "- 4 条 query 必须明显不同，不能只是换同义词；\n"
            "- 不要编造数字；\n"
            "- 只输出 4 行 query。",
        ),
        ("human", "问题：{q}"),
    ]
)

queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
print("queries:")
for q in queries:
    print("-", q)

# 融合策略：RRF（Reciprocal Rank Fusion）
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}

for q in queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)

merged = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)]

print("merged hits:", len(merged))
for i, d in enumerate(merged[:8], 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()


queries:
- 道通2024年核心竞争力
- 道通2024年年报中核心竞争力分析
- 道通2024年经营情况讨论与分析章节的核心竞争力
- 道通2024年主营业务与产品线的核心优势
merged hits: 24
[1] {'h2': '1、资产及负债状况', 'chunk_in_doc': 272, 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'section_id': '道通24年年报__full::section_87', 'h1': '2024 年度报告', 'doc_id': '道通24年年报__full', 'section_title': '1、资产及负债状况', 'source_doc_count': 306, 'parse_source': 'paddleocr_vl', 'page_end': 306, 'doc_group': 'full_document', 'section_in_doc': 87, 'chunk_id': 272, 'source': '道通24年年报', 'page_start': 1, 'content_level': 'chunk', 'chunk_in_section': 12}
style='text-align: center; word-wrap: break-word;'>-0.02</td><td style='text-align: center; word-wrap: break-word;'>2,868.01</td><td style='text-align: center; word-wrap: break-word;'>说明15</td></tr></table>

[2] {'doc_id': '道通24年年报__full', 'section_id': '道通24年年报__full::section_194', 'section_in_doc': 194, 'h3': '(二) 投资者关系及保护', 'page_start': 1, 'page_end': 306, 'chunk_in_doc': 508, 'h1': '2024 年度报告', 'doc_group': 'full_document', 'file

### HyDE

In [30]:
hyde_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是 HyDE 模块。请为用户问题写一段可能出现在年报中的‘假设答案’，用正式书面语，尽量包含可检索的关键词（业务、产品线、收入、分部等）。不要编造具体数值。",
        ),
        ("human", "问题：{q}"),
    ]
)

hypo = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
print("hypo (first 400 chars):\n", hypo[:400])

hits = vs.similarity_search(hypo, k=5)
for i, d in enumerate(hits, 1):
    print(f"[{i}]", d.metadata)
    print(d.page_content[:260].replace("\n", " "))
    print()

hypo (first 400 chars):
 根据道通科技2024年年度报告，本公司的核心竞争力主要体现在以下几个方面：

**1. 持续高强度的研发投入与技术创新能力**
公司始终坚持将技术创新作为发展的核心驱动力，在汽车智能诊断、检测领域及新能源电池领域保持高强度的研发投入。报告期内，公司进一步巩固了在诊断协议、专用芯片、智能算法及云平台方面的技术壁垒，特别是在新能源汽车综合诊断、电池检测与维修技术方面取得了显著进展，相关产品线（如MaxiSYS系列智能诊断设备及新能源电池分析系统）的技术领先性持续增强，为全球汽车后市场提供了先进的数字化智能维修解决方案。

**2. 全球化布局与深厚的客户基础**
公司依托完善的全球营销网络，在北美、欧洲、中国、亚太及其他新兴市场建立了稳定的业务分部。凭借多年积累的品牌声誉、全系列的产品矩阵以及本地化的技术支持服务，公司与众多大型汽车连锁服务商、分销商及专业维修机构建立了长期稳固的合作关系。全球
[1] {'page_end': 306, 'doc_group': 'full_document', 'chunk_in_section': 0, 'doc_id': '道通24年年报__full', 'chunk_in_doc': 490, 'page_start': 1, 'section_in_doc': 179, 'source_doc_count': 306, 'section_id': '道通24年年报__full::section_179', 'h1': '2024 年度报告', 'file_path': 'paddleocr_vl/autel_annual_report_2024.md', 'chunk_id': 490, 'section_title': '2、建立健全内部控制制度，防范公司经营风险', 'source': '道通24年年报', 'h2': '2、建立健全内部控制制度，防范公司经营风险', 'content_level': 'chunk', 'parse_source': 'paddleocr_vl'}
公司根据《中华人民共和国公司法》《中华人民共和国证券法》《企业内部控制基本规范》及配套指引的规定，《上市公司内部控制指引》及其他内部控制监管和相关规定，建立、完善公司经营管理中各环节的风险控制措

### Parent-Child

**核心思想：** 检索需要细粒度（chunk 级别），但 LLM 生成需要大背景（section 级别）。

做法：
1. 用 **chunk-level** collection 做向量检索，命中最相关的 chunk
2. 从命中的 chunk metadata 里拿到 `section_id`
3. 用 `section_id` 去 **section-level** collection 查回完整的 section 文本
4. 把 section 文本交给 LLM，而不是只给小 chunk

这解决了 RAG 里经典的"检索精度 vs. 上下文完整性"矛盾。

In [31]:
# 4) Small-to-Big：在 chunk 上检索，返回 parent section 给 LLM

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def small_to_big_retrieve(query: str, k: int = 5, verbose: bool = True):
    """在 chunk collection 上检索，再通过 section_id 回溯 parent section。"""
    chunk_hits = vs.similarity_search(query, k=k)

    section_ids_seen = set()
    parent_sections = []

    for hit in chunk_hits:
        sid = hit.metadata.get("section_id")
        if not sid or sid in section_ids_seen:
            continue
        section_ids_seen.add(sid)

        section_result = section_vs.get(ids=[sid])
        if section_result and section_result["documents"]:
            parent_sections.append({
                "section_id": sid,
                "section_title": section_result["metadatas"][0].get("section_title", ""),
                "text": section_result["documents"][0],
                "triggered_by_chunks": [
                    h.metadata.get("chunk_in_section")
                    for h in chunk_hits
                    if h.metadata.get("section_id") == sid
                ],
            })

    if verbose:
        print(f"query: {query}")
        print(f"chunk hits: {len(chunk_hits)} → unique parent sections: {len(parent_sections)}\n")
        for i, sec in enumerate(parent_sections, 1):
            print(f"[section {i}] {sec['section_id']}")
            print(f"  title: {sec['section_title']}")
            print(f"  triggered by chunk_in_section: {sec['triggered_by_chunks']}")
            print(f"  text preview: {sec['text'][:300].replace(chr(10), ' ')}")
            print()

    return parent_sections


parent_sections = small_to_big_retrieve(QUESTION, k=5)


query: 根据道通2024年年报来看，其核心竞争力在哪？
chunk hits: 5 → unique parent sections: 5

[section 1] 道通24年年报__full::section_234
  title: (二) 担保情况
  triggered by chunk_in_section: [12]
  text preview: ✓适用 ☐不适用   单位：元币种：人民币   <table border=1 style='margin: auto; word-wrap: break-word;'><tr><td colspan="15">公司对外担保情况（不包括对子公司的担保）</td></tr><tr><td style='text-align: center; word-wrap: break-word;'>担保方</td><td style='text-align: center; word-wrap: break-word;'>担保方与上市公司的关系</td><td style='text-align: cen

[section 2] 道通24年年报__full::section_0
  title: 2024 年度报告
  triggered by chunk_in_section: [0]
  text preview: 深圳市道通科技股份有限公司   <div style="text-align: center;"><img src="道通24年年报_p0001-0050/imgs/img_in_image_box_4_608_1191_1678.jpg" alt="Image" width="99%" /></div>

[section 3] 道通24年年报__full::section_484
  title: (4). 其他应付款
  triggered by chunk_in_section: [3]
  text preview: 按款项性质列示其他应付款   ✓适用 ☐不适用   单位：元币种：人民币   <table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; wo

### 同题回答对比：Baseline / Rewrite / Multi-Query(RRF) / HyDE / Parent-Child

In [33]:
import pandas as pd
from IPython.display import display
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "你是年报分析助手。基于给定检索内容回答问题。"
            "优先给出结构化结论；若证据不足，明确说明缺失信息。\n\n"
            "检索内容：\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def get_doc_title(d):
    return d.metadata.get("section_title") or d.metadata.get("h3") or d.metadata.get("h2") or "无标题"


def build_context_from_docs(docs, max_docs=5, max_chars=1200):
    blocks = []
    for d in docs[:max_docs]:
        blocks.append(f"【{get_doc_title(d)}】\n{d.page_content[:max_chars]}")
    return "\n\n---\n\n".join(blocks)


def build_context_from_sections(sections, max_sections=5, max_chars=2000):
    return "\n\n---\n\n".join(
        f"【{sec['section_title']}】\n{sec['text'][:max_chars]}"
        for sec in sections[:max_sections]
    )


def answer_with_context(context):
    return llm.invoke(
        answer_prompt.format_messages(context=context, question=QUESTION)
    ).content.strip()


# 1) baseline
baseline_docs = vs.similarity_search(QUESTION, k=5)

# 2) rewrite
rewritten_q = llm.invoke(rewrite_prompt.format_messages(q=QUESTION)).content.strip()
rewrite_docs = vs.similarity_search(rewritten_q, k=5)

# 3) multi-query + RRF
multi_queries = [
    s.strip()
    for s in llm.invoke(multi_prompt.format_messages(q=QUESTION)).content.splitlines()
    if s.strip()
]
rrf_k = 60
per_query_k = 8
rrf_scores = {}
doc_by_key = {}
for q in multi_queries:
    docs = vs.similarity_search(q, k=per_query_k)
    for rank, d in enumerate(docs, 1):
        key = (d.metadata.get("chunk_id"), d.page_content[:80])
        doc_by_key[key] = d
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (rrf_k + rank)
multi_docs = [doc_by_key[key] for key in sorted(rrf_scores, key=rrf_scores.get, reverse=True)][:5]

# 4) HyDE
hypo_q = llm.invoke(hyde_prompt.format_messages(q=QUESTION)).content.strip()
hyde_docs = vs.similarity_search(hypo_q, k=5)

# 5) Parent-Child (Small-to-Big)
parent_sections = small_to_big_retrieve(QUESTION, k=5, verbose=False)
parent_context = build_context_from_sections(parent_sections)

records = [
    {
        "strategy": "Baseline",
        "retrieval_query": QUESTION,
        "hits": len(baseline_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in baseline_docs),
        "answer": answer_with_context(build_context_from_docs(baseline_docs)),
    },
    {
        "strategy": "Rewrite",
        "retrieval_query": rewritten_q,
        "hits": len(rewrite_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in rewrite_docs),
        "answer": answer_with_context(build_context_from_docs(rewrite_docs)),
    },
    {
        "strategy": "Multi-Query (RRF)",
        "retrieval_query": " | ".join(multi_queries),
        "hits": len(multi_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in multi_docs),
        "answer": answer_with_context(build_context_from_docs(multi_docs)),
    },
    {
        "strategy": "HyDE",
        "retrieval_query": hypo_q[:220].replace("\n", " ") + ("..." if len(hypo_q) > 220 else ""),
        "hits": len(hyde_docs),
        "hit_titles": " | ".join(get_doc_title(d) for d in hyde_docs),
        "answer": answer_with_context(build_context_from_docs(hyde_docs)),
    },
    {
        "strategy": "Parent-Child",
        "retrieval_query": QUESTION,
        "hits": len(parent_sections),
        "hit_titles": " | ".join(sec["section_title"] for sec in parent_sections),
        "answer": answer_with_context(parent_context),
    },
]

comparison_df = pd.DataFrame(records)
comparison_df.insert(0, "question", QUESTION)

pd.set_option("display.max_colwidth", 180)
display(comparison_df)


,question,strategy,retrieval_query,hits,hit_titles,answer
0,根据道通2024年年报来看，其核心竞争力在哪？,Baseline,根据道通2024年年报来看，其核心竞争力在哪？,5,(二) 担保情况 | 2024 年度报告 | (4). 其他应付款 | 59、盈余公积 | 1、负责任供应链,根据您提供的检索内容，**无法直接分析**道通科技2024年度的核心竞争力。\n\n**原因如下：**\n您提供的检索内容主要涉及公司治理（担保情况）、财务报表科目（其他应付款、盈余公积调整）及供应链管理理念，这些信息**不足以**支撑对“核心竞争力”这一关键战略问题的分析。\n\n**核心竞争力**通常体现在公司的技术优势、产品力、品牌价值、市场地...
1,根据道通2024年年报来看，其核心竞争力在哪？,Rewrite,道通2024年年报中关于核心竞争力的分析,5,（一）公司实际控制人、股东、关联方、收购人以及公司等承诺相关方在报告期内或持续到报告期内的承诺事项 | 三、重大风险提示 | 4、处置子公司 | 三、股东大会情况简介 | (二) 担保情况,根据您提供的检索内容，**无法直接得出道通科技2024年核心竞争力的具体信息**。\n\n**原因说明：**\n您提供的检索内容主要涉及公司治理、承诺事项、风险提示、子公司处置、股东大会及担保情况，这些部分通常不用于阐述公司的核心竞争力。\n\n**核心竞争力的信息通常出现在年报的以下章节：**\n* **第三节 管理层讨论与分析**：其中“报告...
2,根据道通2024年年报来看，其核心竞争力在哪？,Multi-Query (RRF),道通2024年核心竞争力 | 道通2024年年报核心竞争力章节 | 道通2024年经营情况讨论与分析中的核心竞争力 | 道通2024年主营业务与产品线的核心优势,5,1、资产及负债状况 | 75、营业外支出 | (二) 投资者关系及保护 | 13、应收账款 | (二) 担保情况,根据您提供的检索内容，**无法直接得出道通科技2024年核心竞争力的具体结论**。\n\n**原因如下：**\n您提供的检索内容均为年报中的具体数据或事实片段，主要涉及资产、负债、营业外支出、投资者关系、应收账款账龄和担保情况。这些信息**并未包含**关于公司“核心竞争力”的论述，例如：\n* 技术研发实力（如专利数量、研发投入占比）\n* ...
3,根据道通2024年年报来看，其核心竞争力在哪？,HyDE,根据道通科技2024年年报，公司核心竞争力主要体现在以下几个方面： 1. **技术创新与研发能力**：公司持续加大研发投入，聚焦汽车智能诊断、检测及新能源领域，形成了覆盖传统燃油车及新能源汽车的完整产品线，尤其在电池检测、智能诊断软件及云平台技术上保持行业领先。 2. **全球化业务布局**：公司通过深化海外分部运营，在北美、欧洲、亚太等关键...,5,3、报告期内新技术、新产业、新业态、新模式的发展情况和未来发展趋势 | （一）公司实际控制人、股东、关联方、收购人以及公司等承诺相关方在报告期内或持续到报告期内的承诺事项 | （四）全球供应链前瞻布局，积极应对贸易风险 | （三）空地一体集群智慧解决方案——开启 AI+机器人业务，第三发展曲线应运而生 | (七) 宏观环境风险,根据您提供的道通2024年年报检索内容，其核心竞争力可归纳为以下三个结构化维度：\n\n### 一、技术研发与创新布局\n1. **AI与机器人技术前沿卡位**：\n * **AI Agent（软件智能体）**：公司已构建AI智算中心（Autel Maxwell AI Cluster），具备AI模型工程化、数据管理、模型训练及Agent开...
4,根据道通2024年年报来看，其核心竞争力在哪？,Parent-Child,根据道通2024年年报来看，其核心竞争力在哪？,5,(二) 担保情况 | 2024 年度报告 | (4). 其他应付款 | 59、盈余公积 | 1、负责任供应链,根据您提供的2024年年报检索内容，**无法直接分析出道通科技的核心竞争力**。\n\n**原因说明：**\n您提供的检索内容主要涉及**担保情况、其他应付款、盈余公积调整**等具体财务和交易细节，以及一份未显示具体内容的图片。这些信息属于公司经营活动的**局部和结果性数据**，而非用于阐述公司核心竞争力的**战略性、资源性、能力性描述**。\n\n...


In [ ]:
# Debug：显式查看不同 query 的 embedding / raw distances / top hits

import pandas as pd
from IPython.display import display


def inspect_query(query_name: str, query_text: str, n_results: int = 5):
    query_vec = emb.embed_query(query_text)
    raw = vs._collection.query(
        query_embeddings=[query_vec],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    rows = []
    for i, (doc, meta, dist) in enumerate(
        zip(raw["documents"][0], raw["metadatas"][0], raw["distances"][0]),
        1,
    ):
        rows.append(
            {
                "query_name": query_name,
                "query_text": query_text,
                "embedding_dim": len(query_vec),
                "rank": i,
                "distance": round(float(dist), 6),
                "section_title": (meta or {}).get("section_title") or (meta or {}).get("h3") or (meta or {}).get("h2"),
                "chunk_id": (meta or {}).get("chunk_id"),
                "preview": doc[:160].replace("\n", " "),
            }
        )
    return rows


queries_to_debug = [("baseline", QUESTION)]

if "rewritten" in globals():
    queries_to_debug.append(("rewrite", rewritten))

if "queries" in globals():
    for idx, q in enumerate(queries[:4], 1):
        queries_to_debug.append((f"multi_{idx}", q))

if "hypo" in globals():
    queries_to_debug.append(("hyde", hypo[:500]))

all_rows = []
for name, text in queries_to_debug:
    all_rows.extend(inspect_query(name, text, n_results=5))

debug_df = pd.DataFrame(all_rows)
pd.set_option("display.max_colwidth", 200)
display(debug_df)
